# Figure: Solubilities during degassing

In [ ]:
from pathlib import Path
import numpy as np

results_directory = Path().resolve().parent / "Model_Outputs"
SAVE_FIG = True

## Import data and styling

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from helpers.plot_styles import (
    PLOTLY_TICK_LEN,
    PLOTLY_FONT,
    PLOTLY_LEGEND_FONTSIZE,
    PLOTLY_TICK_FONTSIZE,
    SAMPLE_DISPLAY_NAMES,
    TOOL_COLORS_HEX,
    TOOL_LINE_STYLE,
)
from helpers.degassing_data import load_all_systems

# --- USER INPUTS --- #
SAMPLES = ["MORB", "Kilauea", "Fuego", "Fogo"]
TOOLS  = ["DCompress", "DCompress (IM)", "EVo", "MAGEC", "SulfurX", "VolFe", "VESIcal_Iacono"]

In [ ]:
systems = load_all_systems(SAMPLES, TOOLS, results_dir=results_directory)

## Build the figure

In [ ]:
# Row definitions: (DataFrame column, y-axis label).
Y_ROWS = [
    ("H2O","log<sub>10</sub>[H'<sub>H<sub>2</sub>O</sub>]",-2.5,-1.6),
    ("CO2", "log<sub>10</sub>[H'<sub>CO<sub>2</sub></sub>]",-0.6,0.5),
    ("S2m","log<sub>10</sub>[H'<sub>S<sup>red</sub></sub>]",-14,-8),
    ("S6p","log<sub>10</sub>[H'<sub>S<sup>ox</sup></sub>]",4,9)
]

n_rows, n_cols = len(Y_ROWS), len(SAMPLES)
top_titles = [SAMPLE_DISPLAY_NAMES.get(s, s) for s in SAMPLES]
subplot_titles = top_titles + [""] * ((n_rows - 1) * n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    shared_xaxes=True, vertical_spacing=0.03, horizontal_spacing=0.05,
    subplot_titles=subplot_titles,
)

for r, (species,y_label,min,max) in enumerate(Y_ROWS, start=1):
    for c, sample in enumerate(SAMPLES, start=1):
        for tool in TOOLS:
            df = systems.get(sample, {}).get(tool)
            if df is None or "P_bars" not in df.columns:
                continue
            p_init = df["P_bars"].iloc[0]
            if p_init == 0:
                continue
            x_norm = df["P_bars"]# / p_init
            if species == 'H2O':
                solubility = np.log10((df['H2OT_m_wtpc']**2.)/(df['H2O_v_mf']*df['P_bars']))
            elif species == 'CO2':
                solubility = np.log10((df["CO2T_m_ppmw"])/(df['CO2_v_mf']*df['P_bars']))
            elif species == 'S2m':
                solubility = np.log10((df['ST_m_ppmw']*(1.-df['S6St_m'])*((10.**df['logfO2'])**1.5/(df['SO2_v_mf']*df['P_bars']))))
            elif species == 'S6p':
                solubility = np.log10((df['ST_m_ppmw']*df['S6St_m'])/(((10.**df['logfO2'])**0.5)*(df['SO2_v_mf']*df['P_bars'])))
            fig.add_trace(
                go.Scatter(
                    mode="lines",
                    x=x_norm, y=solubility,
                    name=tool,
                    line=dict(color=TOOL_COLORS_HEX.get(tool, "#333"), width=2,
                              dash=TOOL_LINE_STYLE.get(tool, "solid"),
                              ),
                    showlegend=(r == 1 and c == 1),
                ),
                row=r, col=c,
            )
        fig.update_yaxes(title_text=y_label if c == 1 else None, range=[min,max], row=r, col=c)
        if r == n_rows:
            fig.update_xaxes(title_text="Pressure (bar)", row=r, col=c, range=[0, None])

legend_style_dict = dict(
    font=dict(size=PLOTLY_LEGEND_FONTSIZE),
    x=0.99, y=0.15,
    xanchor="right", yanchor="bottom",
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
)

fig.update_layout(
    height=800, width=1000,
    plot_bgcolor="white",
    margin=dict(t=40, r=30, l=60, b=50),
    font=PLOTLY_FONT,
    legend=legend_style_dict,
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
    rangemode="tozero",
)

if SAVE_FIG:
    fig.write_image("figures/Fig_solubilities.png", scale=2, height=800, width=1000)

fig.show()